# อ่าน Iceberg `fact_sales_enriched` จาก S3 ด้วย AWS Glue

- ตาราง: `com7_gold.fact_sales_enriched` (Glue Data Catalog · Iceberg v2 · ap-southeast-7)
- ที่เก็บ: `s3://com7data-gold-603238661233/detail-sale/fact_sales_enriched/` → `data/` + `metadata/`
- partition: `sale_year`, `sale_month` (identity)

> ⚠️ **ห้ามอ่าน `data/` ตรงด้วย `spark.read.parquet`** — ตารางใช้จริง 334 ไฟล์ (3.39 GB) แต่ในโฟลเดอร์มี ~9,351 ไฟล์ (108 GB)
> เพราะ job สร้างตารางใหม่ทุกคืนแล้วไม่ลบไฟล์เก่า อ่านตรงจะได้ข้อมูลซ้ำหลายสิบเท่า · ต้องอ่านผ่าน Iceberg เสมอ (cell 3 หรือ 6)

ใช้กับ **Glue Studio notebook** หรือ Jupyter ที่ติดตั้ง `aws-glue-sessions` kernel

## 1. ตั้งค่า session (ต้องรันก่อน cell อื่น)

In [ ]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%region ap-southeast-7

In [ ]:
%%configure
{
  "--datalake-formats": "iceberg",
  "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://com7data-gold-603238661233/ --conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO"
}

## 2. เริ่ม Spark

In [ ]:
import sys
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

BUCKET = "com7data-gold-603238661233"
TABLE_PATH = f"s3://{BUCKET}/detail-sale/fact_sales_enriched"
FQ = "glue_catalog.com7_gold.fact_sales_enriched"
print(spark.version)

## 3. อ่านผ่าน Glue Catalog (วิธีหลัก)

In [ ]:
df = spark.table(FQ)          # lazy · ยังไม่อ่านข้อมูล
df

## 4. ดู schema

In [ ]:
df.printSchema()

In [ ]:
# schema เป็นตาราง: ชื่อ · type · nullable
import pandas as pd
pd.DataFrame([(f.name, f.dataType.simpleString(), f.nullable) for f in df.schema.fields],
             columns=["column", "type", "nullable"])

In [ ]:
# DESCRIBE แบบ Iceberg — เห็น partitioning, location, format, properties
spark.sql(f"DESCRIBE TABLE EXTENDED {FQ}").show(60, truncate=False)

## 5. metadata ของ Iceberg (snapshot · partition · ไฟล์)

In [ ]:
# snapshot ทั้งหมด — ตารางนี้ถูกสร้างใหม่ทุกคืน จะเห็นแค่ 1 snapshot
spark.sql(f"SELECT committed_at, snapshot_id, operation, summary['total-records'] AS records, summary['total-data-files'] AS files FROM {FQ}.snapshots").show(truncate=False)

In [ ]:
# จำนวนแถว / ไฟล์ ต่อ partition — อ่านจาก metadata ไม่สแกนข้อมูล
spark.sql(f"""
SELECT partition.sale_year, partition.sale_month, record_count, file_count,
       ROUND(total_data_file_size_in_bytes / 1e6, 1) AS size_mb
FROM {FQ}.partitions
ORDER BY 1, 2
""").show(60)

In [ ]:
# ไฟล์ที่ตารางใช้จริง — เทียบกับจำนวนไฟล์ในโฟลเดอร์ data/ บน S3
spark.sql(f"SELECT COUNT(*) AS files, ROUND(SUM(file_size_in_bytes)/1e9, 2) AS gb FROM {FQ}.files").show()

## 6. (ทางเลือก) อ่านจาก path โดยไม่ผ่าน Catalog
ใช้เมื่อตารางยังไม่ได้ลงทะเบียน · ต้องชี้ไปที่ไฟล์ `*.metadata.json` ล่าสุด (ไม่มี `version-hint.text` เพราะตารางสร้างผ่าน Glue Catalog)

In [ ]:
import boto3
s3 = boto3.client("s3")
keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix="detail-sale/fact_sales_enriched/metadata/"):
    keys += [(o["LastModified"], o["Key"]) for o in page.get("Contents", []) if o["Key"].endswith(".metadata.json")]
latest = max(keys)[1]                       # ใช้เวลาแก้ไข ไม่ใช่เลขหน้าชื่อ — ตารางนี้เลขวนกลับเป็น 00000 ทุกคืน
metadata_path = f"s3://{BUCKET}/{latest}"
print(metadata_path)

df_path = spark.read.format("iceberg").load(metadata_path)
df_path.printSchema()

## 7. (ทางเลือก) ดู schema จาก metadata JSON โดยไม่ใช้ Spark
เร็วและไม่เสียค่า Glue — ใช้ได้แม้ใน Jupyter ธรรมดาที่มีแค่ boto3

In [ ]:
import json
meta = json.loads(s3.get_object(Bucket=BUCKET, Key=latest)["Body"].read())
schema = next(s for s in meta["schemas"] if s["schema-id"] == meta["current-schema-id"])
print("format-version:", meta["format-version"])
print("partition spec:", [(p["name"], p["transform"]) for p in meta["partition-specs"][-1]["fields"]])
pd.DataFrame([(f["id"], f["name"], f["type"], f["required"]) for f in schema["fields"]],
             columns=["field_id", "column", "type", "required"])

## 8. ตัวอย่าง query — กรอง partition ก่อนเสมอ

In [ ]:
sep = (df.filter((F.col("sale_year") == 2026) & (F.col("sale_month") == 9))   # อ่านแค่ partition นี้
         .groupBy("region", "category")
         .agg(F.sum("sale_amt").alias("sales"), F.sum("gp_amt").alias("gp"), F.count("*").alias("rows"))
         .orderBy(F.desc("sales")))
sep.show(20, truncate=False)

In [ ]:
# ตรวจว่า Iceberg ตัด partition จริง — มองหา filter sale_year/sale_month ในส่วน BatchScan
sep.explain()

In [ ]:
# ย้อนดูข้อมูล ณ snapshot ก่อนหน้า (time travel) — ตารางนี้สร้างใหม่ทุกคืนจึงมี snapshot เดียว ใช้ได้เมื่อเปลี่ยนเป็น MERGE/overwrite แบบไม่ drop
# spark.read.option("snapshot-id", <id>).table(FQ)

## 9. ปิด session (ไม่งั้นคิดเงินจนครบ idle_timeout)

In [ ]:
%stop_session